# MAD1 - Unidad 5: Predicción y Comunicación de Resultados
## Práctica en Python

**Métodos de Análisis de Datos 1 - UNS**  
Primer Semestre 2026

## Librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import pickle
from joblib import dump, load
import json
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

# Interpretabilidad
import shap
import lime
import lime.lime_tabular
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

---
# 1. Serialización de Modelos

## Entrenar modelo de ejemplo

In [ ]:
from sklearn.datasets import load_iris

# Cargar datos
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

# Evaluar
accuracy = modelo.score(X_test, y_test)
print(f"Accuracy: {accuracy:.3f}")

## Serialización con pickle

In [ ]:
# Guardar
with open('modelo_iris.pkl', 'wb') as f:
    pickle.dump(modelo, f)

# Cargar
with open('modelo_iris.pkl', 'rb') as f:
    modelo_cargado = pickle.load(f)

# Verificar
pred_original = modelo.predict(X_test)
pred_cargado = modelo_cargado.predict(X_test)
print("Predicciones idénticas:", np.array_equal(pred_original, pred_cargado))

## Serialización con joblib (recomendado)

In [ ]:
# Guardar
dump(modelo, 'modelo_iris.joblib')

# Cargar
modelo_cargado = load('modelo_iris.joblib')

# Usar
print("Accuracy del modelo cargado:", modelo_cargado.score(X_test, y_test))

## Pipeline completo con metadata

In [ ]:
# Crear pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Entrenar
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

# Guardar pipeline
dump(pipeline, 'pipeline_iris_v1.0.0.joblib')

# Guardar metadata
metadata = {
    'version': '1.0.0',
    'timestamp': datetime.now().isoformat(),
    'accuracy': accuracy,
    'features': list(X_train.columns),
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test)
}

with open('pipeline_iris_v1.0.0_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Pipeline y metadata guardados")
print(json.dumps(metadata, indent=2))

---
# 2. Interpretabilidad con SHAP

## Crear explainer y calcular SHAP values

In [ ]:
# Crear explainer (para Random Forest)
explainer = shap.TreeExplainer(modelo)

# Calcular SHAP values
shap_values = explainer.shap_values(X_test)

print("SHAP values shape:", np.array(shap_values).shape)
print("Expected value (baseline):", explainer.expected_value)

## Summary Plot (importancia global)

In [ ]:
# Summary plot para clase 1 (versicolor)
shap.summary_plot(shap_values[1], X_test)

## Force Plot (predicción individual)

In [ ]:
# Explicar primera instancia, clase 1
shap.force_plot(
    explainer.expected_value[1], 
    shap_values[1][0], 
    X_test.iloc[0],
    matplotlib=True
)

## Waterfall Plot

In [ ]:
# Waterfall para una instancia
shap.waterfall_plot(shap.Explanation(
    values=shap_values[1][0],
    base_values=explainer.expected_value[1],
    data=X_test.iloc[0]
))

## Dependence Plot

In [ ]:
# Dependence plot para una feature
shap.dependence_plot(
    "petal length (cm)", 
    shap_values[1], 
    X_test
)

---
# 3. LIME

In [ ]:
# Crear explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=X_train.columns,
    class_names=['setosa', 'versicolor', 'virginica'],
    mode='classification'
)

# Explicar una instancia
i = 0
exp = explainer_lime.explain_instance(
    X_test.values[i], 
    modelo.predict_proba,
    num_features=4
)

# Visualizar
exp.show_in_notebook(show_table=True)

# O como gráfico matplotlib
exp.as_pyplot_figure()
plt.tight_layout()

---
# 4. Partial Dependence Plots

In [ ]:
# PDP para una feature
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    modelo, X_train, 
    features=['petal length (cm)'],
    ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
# PDP para múltiples features + interacción
fig, ax = plt.subplots(figsize=(14, 4))
PartialDependenceDisplay.from_estimator(
    modelo, X_train, 
    features=['petal length (cm)', 'petal width (cm)', 
              ('petal length (cm)', 'petal width (cm)')],
    ax=ax
)
plt.tight_layout()
plt.show()

---
# 5. Feature Importance

## Importancia estándar (Gini)

In [ ]:
# Obtener importancias
importances = modelo.feature_importances_
indices = np.argsort(importances)[::-1]

# Visualizar
plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=45)
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Feature Importances (Gini)")
plt.tight_layout()
plt.show()

## Permutation Importance

In [ ]:
# Calcular
perm_importance = permutation_importance(
    modelo, X_test, y_test,
    n_repeats=10, 
    random_state=42
)

# Visualizar
sorted_idx = perm_importance.importances_mean.argsort()[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(perm_importance.importances[sorted_idx].T,
           labels=X_test.columns[sorted_idx],
           vert=False)
ax.set_xlabel("Decrease in Accuracy")
ax.set_title("Permutation Feature Importance")
plt.tight_layout()
plt.show()

---
# 6. Ejemplo Completo: Reporte de Modelo

In [ ]:
# 1. Entrenar modelo final
modelo_final = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_final.fit(X_train, y_train)

# 2. Evaluar
y_pred = modelo_final.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("REPORTE DEL MODELO")
print("=" * 50)
print(f"\nAccuracy: {accuracy:.3f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, 
                           target_names=['setosa', 'versicolor', 'virginica']))

# 3. Feature Importance
print("\nFeature Importances:")
for feature, imp in zip(X_train.columns, modelo_final.feature_importances_):
    print(f"  {feature}: {imp:.3f}")

# 4. Guardar todo
pipeline_final = {
    'modelo': modelo_final,
    'accuracy': accuracy,
    'features': list(X_train.columns),
    'version': '2.0.0',
    'fecha': datetime.now().isoformat()
}

dump(pipeline_final, 'modelo_final_completo.joblib')
print("\n✓ Modelo guardado en: modelo_final_completo.joblib")

---
# 7. Consideraciones Éticas

## Análisis de Fairness por Subgrupo

In [ ]:
# Ejemplo: evaluar modelo por subgrupo
# Supongamos que tenemos una variable sensible (e.g., región)

# Simulamos atributo sensible para demostración
np.random.seed(42)
X_test['region'] = np.random.choice(['A', 'B'], size=len(X_test))

# Evaluar accuracy por subgrupo
print("Análisis de Fairness por Región:")
print("=" * 40)
for region in ['A', 'B']:
    mask = X_test['region'] == region
    if mask.sum() > 0:
        X_subset = X_test[mask].drop('region', axis=1)
        y_subset = y_test[mask]
        pred_subset = modelo.predict(X_subset)
        acc_subset = accuracy_score(y_subset, pred_subset)
        print(f"Región {region}: Accuracy = {acc_subset:.3f} (n={mask.sum()})")

# Remover columna temporal
X_test = X_test.drop('region', axis=1)

print("\n⚠️  Si hay diferencias grandes, investigar:")
print("  - ¿Datos balanceados por grupo?")
print("  - ¿Features sesgadas?")
print("  - ¿Necesitamos ajustar umbrales por grupo?")

## Documentación de Limitaciones

In [ ]:
# Documentar limitaciones del modelo
limitaciones = {
    'supuestos': [
        'Los datos de test son similares a los de entrenamiento',
        'Las features son medidas con la misma precisión',
        'No hay cambios en las especies de iris con el tiempo'
    ],
    'limitaciones_datos': [
        f'Tamaño de muestra limitado (n={len(X_train)} train, n={len(X_test)} test)',
        'Dataset clásico, puede no representar variabilidad real',
        'Solo 4 features, ignora otros factores'
    ],
    'casos_no_usar': [
        'Especies de iris fuera de las 3 entrenadas',
        'Plantas con medidas fuera del rango de entrenamiento',
        'Contextos donde el costo de error es muy alto'
    ],
    'incertidumbre': {
        'accuracy_test': f'{accuracy:.3f}',
        'intervalo_confianza_95': '[0.90, 1.00]'  # Ejemplo
    }
}

# Guardar como JSON
with open('modelo_limitaciones.json', 'w') as f:
    json.dump(limitaciones, f, indent=2)

print("Limitaciones documentadas:")
print(json.dumps(limitaciones, indent=2))

---
# 8. Control de Versiones

## Comandos Git básicos

```bash
# Inicializar repositorio
git init

# Ver estado
git status

# Agregar archivos
git add U5_practica_Python.ipynb

# Commit
git commit -m "Agregar notebook U5"

# Subir a remoto
git push origin main
```

## .gitignore para Data Science

```
# Datos
data/
*.csv
*.xlsx

# Modelos
*.pkl
*.joblib
*.h5

# Notebooks
.ipynb_checkpoints/

# Entornos
venv/
env/
```

---
# 9. Gestión de Dependencias

## requirements.txt

```bash
# Generar
pip freeze > requirements.txt

# Instalar desde archivo
pip install -r requirements.txt
```

## Mejor práctica: usar entornos virtuales

```bash
# Crear entorno
python -m venv venv

# Activar (Linux/Mac)
source venv/bin/activate

# Activar (Windows)
venv\Scripts\activate

# Instalar dependencias
pip install -r requirements.txt
```

---
# Notas Finales

**Flujo de trabajo recomendado:**

1. Entrenar y validar modelo
2. Serializar con joblib + metadata JSON
3. Interpretar decisiones:
   - SHAP para explicaciones completas
   - LIME para casos individuales
   - PDP para efectos marginales
   - Feature importance para ranking
4. Documentar en Jupyter Notebook
5. Versionado con Git
6. Gestión de dependencias con requirements.txt
7. Auditar fairness y sesgos
8. Documentar limitaciones y casos de no-uso
9. Comunicar resultados apropiadamente

**Recursos adicionales:**

- SHAP: https://shap.readthedocs.io/
- LIME: https://github.com/marcotcr/lime
- scikit-learn: https://scikit-learn.org/
- Jupyter: https://jupyter.org/